In [ ]:
import numpy as np
import heapq
import matplotlib.pyplot as plt
import random
import os

In [2]:
def clear_variables():
    """Clears only user-defined variables while keeping functions, classes, and imports."""
    protected = {
        name for name, obj in globals().items()
        if name.startswith('__') or
           callable(obj) or
           isinstance(obj, type) or
           name in ['np', 'plt', 'heapq', 'random', 'clear_variables','os']  # Add your imports and this function itself
    }

    # Delete everything not in the protected set
    to_delete = [name for name in globals() if not name.startswith('_') and name not in protected]
    for name in to_delete:
        del globals()[name]
    
    # Delete them
    for name in to_delete:
        del globals()[name]

def oracle_select(ucb_values, k):
    """Selects top-k arms using UCB values."""
    return heapq.nlargest(k, range(len(ucb_values)), key=lambda i: ucb_values[i])

def expect_reward(mu,super_arm):
    product = 1
    for i in super_arm:
        product *= (1 - mu[i]) # must be mu_on
    return 1 - product

def trigger(super_arm,mu_on):
    mu_update = np.zeros(len(super_arm))# X_i,t
    observed_arm = []
    for i in super_arm:
        observed_arm.append(i)
        if random.random() < mu_on[i]: # X_i,t == 1
            mu_update[len(observed_arm) - 1] = 1
            return mu_update,observed_arm
    return mu_update,observed_arm # if 0, it is no arm searched out

In [3]:
def offline(m,mu_off,num_each_arm):
    delta = 0.1
    sample = np.zeros(m)
    log_CLCB = np.log(np.divide(2*m*num_each_arm ,delta))# n in paper: 一个臂拉取多少次，fix
    mu_hat_off = np.zeros(m)
    LCB = np.zeros(m)
    choose = [np.random.randint(150, num_each_arm + 1) for i in range(m)]
    for i in range(m):
        sample[i] = np.random.binomial(choose[i],mu_off[i])
        mu_hat_off[i] = np.divide(sample[i],num_each_arm)
        LCB[i] = mu_hat_off[i]- np.sqrt(np.divide(log_CLCB,2* num_each_arm))
        
    return choose,mu_hat_off, LCB

def MeanRewardOff(m, mu_off, k, mu_on,num_each_arm,off_turn):
    all_rewards = []
    all_N = []         
    all_mu_hat_off = []
    
    for t in range(off_turn):
        N, mu_hat_off ,LCB= offline(m, mu_off,num_each_arm)
        super_arm = oracle_select(mu_hat_off, k)

        reward = expect_reward(mu_on, super_arm)
        # print(f"reward:{reward}")
        
        all_rewards.append(reward)
        all_N.append(N)
        all_mu_hat_off.append(mu_hat_off)
    
    # 计算均值
    mean_reward = np.mean(all_rewards)
    mean_N = np.mean(all_N, axis=0)  # axis = 0: do summation to row vector
    mean_mu_hat_off = np.mean(all_mu_hat_off, axis=0)
    
    return mean_reward, mean_N, mean_mu_hat_off
# def RewardOff(m, mu_off, k, mu_on,num_each_arm,off_turn):
#     N_off , mu_hat_off,LCB = offline(m, mu_off,num_each_arm)
#     print(f"mu_hat_off:{mu_hat_off}")
#     print(f"LCB:{LCB}")
#     super_arm = oracle_select(LCB, k)
#     print(f"super_arm_off:{super_arm}")
#     reward = expect_reward(mu_on, super_arm)
#     # print(f"reward:{reward}")
#     return reward, N_off,mu_hat_off

In [4]:
def hybrid(m,k,t,N_online,mu_on,mu_hat_on,N_off,mu_hat_off,V):
    ucb_online = np.zeros(m)
    ucb_hybrid = np.zeros(m)
    
    for i in range(m):
        # Pure online UCB
        # log_term = np.log(4 * m * t)
        log_term = np.log(100)

        # N_online
        if N_online[i] == 0:
            ucb_online[i] = 1
        else:
            ucb_online[i] = mu_hat_on[i] + np.sqrt(2 * log_term / (N_online[i]))

        # N[i] + N_online[i] == 0
        if N_off[i] + N_online[i] == 0:
            ucb_hybrid[i] = 1
        else:
            ucb_hybrid[i] = np.divide(N_off[i]*mu_hat_off[i] + N_online[i]* mu_hat_on[i], N_off[i] + N_online[i]) + np.sqrt(2 * log_term / (N_online[i]+N_off[i])) + np.divide(N_off[i]*V[i],N_off[i] + N_online[i])
            
    # ucb = np.minimum(ucb_online, ucb_hybrid)
    ucb = np.minimum(np.minimum(ucb_online, ucb_hybrid), np.ones(m))
    print(f"ucb={ucb}")

    ## Trigger part
    # super_arm = oracle_select(ucb,k)
    super_arm = oracle_select(ucb,k)
    print(f"hybrid super_arm:{super_arm}")

    reward_online = expect_reward(mu_on,super_arm)

    mu_update, observed_arm = trigger(super_arm,mu_on)


    # Reward part
    for i in observed_arm: 
        N_online[i] += 1

    if mu_update.sum() != 0:
        for idx, arm in enumerate(observed_arm):
            if idx == len(observed_arm) - 1: # chosen one
                mu_hat_on[arm] += np.divide(1 - mu_hat_on[arm],N_online[arm])
                continue
            mu_hat_on[arm] += np.divide(0 - mu_hat_on[arm],N_online[arm]) # triggered but not chosen

    else:
        for arm in observed_arm: # all did not chosen by observed
            mu_hat_on[arm] += np.divide(0 - mu_hat_on[arm],N_online[arm])
    return reward_online,N_online,mu_hat_on



In [5]:
def single_run_hybrid(m, k, T, reward_star, mu_on,N_off, mu_hat_off, V1):

    # 1.initial
    N_online,mu_hat_on = np.zeros(m),np.zeros(m)
    N_biased,mu_hat_biased  = np.zeros(m),np.zeros(m)
    gap_online = np.zeros(T)
    gap_hybrid_biased = np.zeros(T)
    cumulative_online = 0.0
    cumulative_hybrid = 0.0

    # 2. running 
    for t in range(1, T+1):
        print("--------------------------------------------------")
        print(f"t={t}")
        reward_online,N_online,mu_hat_on= hybrid(m,k,t,N_online,mu_on,mu_hat_on,np.zeros(m),mu_hat_off=np.zeros(m),V = np.zeros(m))
        cumulative_online += (reward_star - reward_online)
        gap_online[t - 1] = cumulative_online

        reward_bias,N_biased,mu_hat_biased= hybrid(m,k,t,N_biased,mu_on,mu_hat_biased,N_off,mu_hat_off,V1)
        cumulative_hybrid += (reward_star - reward_bias)
        gap_hybrid_biased[t - 1] = cumulative_hybrid        
    return gap_online , gap_hybrid_biased

In [6]:
from concurrent.futures import ProcessPoolExecutor
from multiprocessing import Pool, cpu_count

from numpy.random import f
clear_variables()
print("Clear up")
m = 10
k = 5
num_trials = 5
mu_on = np.linspace(0.4,0.6,m)
save_dir = r"D:\Context_Bandit\data"
# mu_on = np.random.uniform(0.3,0.7,m) # Adjust : 0 can be changed to 0.5,0.6
# mu_on[0] = 0.699

# for bias in {0.2, 0.3, 0.4}:
for bias in {0.4}:
    # bias generation
    while True:
        V = [bias] * m # Bias bound
        count = 0
        for i in range(m):
            if random.random() < 0.5:
                V[i] = -V[i]
                count += 1
        if m / 2 - 5 <= count <= m / 2 + 5:
            break


    # V = [bias] * m # Bias bound
    # for i in range(m):
    #     if random.random() < 0.5:
    #         V[i] = -V[i]


    print(f"V={V}")


    mu_off =  mu_on + V 
    print(f"mu_off:{mu_off}")

    V1 = [bias] * m

    # 2.Find the oracle 
    super_arm_oracle = oracle_select(mu_on,k)
    reward_star = expect_reward(mu_on,super_arm_oracle)

    # 3. do offline, online, hybrid-unbiased, hybrid-biased
    ## 3.1 offline
    reward_off = 0
    reward = []
    num_each_arm = 200
    reward_off,N_off,mu_hat_off = MeanRewardOff(m,mu_off,k,mu_on,num_each_arm,off_turn= 1)# 5000 is not good, maybe it's because of the seed
    T = 8000
    gap_offline = (reward_star - reward_off ) * range(1,T+1)
    gap_off_array = np.array(gap_offline)
    np.save(os.path.join(save_dir, f'gap_off_{bias}.npy'), gap_off_array)
    print("Off is done!")

    gap_online = np.zeros(T)
    gap_hybrid_biased = np.zeros(T)
    gap_on = []
    gap_hb = []

    # 3.2 online, hybrid-unbiased, hybrid-biased

    # Run the simulation 5 times

    N_off = [num_each_arm] * m

    gap_online , gap_hybrid_biased = single_run_hybrid(m,k,T,reward_star,mu_on,N_off, mu_hat_off ,V1)
    print(f"gap_online:{gap_online}")
    print(f"gap_hybrid_biased:{gap_hybrid_biased}")

    # N = [num_each_arm] * m
    # args = (m,k,T,reward_star,mu_on,N,mu_hat_off,V1)
    # tasks = [args] * num_trials
    # num_processes = cpu_count() - 1

    # with Pool(processes=num_processes) as pool:
    #     results = pool.starmap(single_run_hybrid, tasks)


    gap_online_array = np.array(gap_online)
    np.save(os.path.join(save_dir, f'gap_online_{bias}.npy'), gap_online_array)
    print("Online is done!")
    gap_hybrid_biased_array = np.array(gap_hybrid_biased)
    np.save(os.path.join(save_dir, f'gap_hb_{bias}.npy'), gap_hybrid_biased_array)
    print("Hybrid-biased is done!")


    # 4.Plot
    plt.plot(gap_offline, color='green', label='CLCB')
    plt.plot(gap_online, color='blue', label='CUCB')
    plt.plot(gap_hybrid_biased, color='crimson', label='H-CUCB')

    plt.xlim(0,T)
    plt.ylim(0,300)
    plt.xlabel("Time Steps (t)")
    plt.ylabel("Cumulative Regret")
    plt.title(f"V={bias},N={num_each_arm}")

    # Customize the legend location and appearance
    plt.legend(loc='upper left', frameon=True, shadow=True)

    plt.grid(True, linestyle='--', alpha=0.7)
    plt.savefig(rf"D:\Context_Bandit\picture\{bias}.pdf", bbox_inches='tight', dpi=300)

    plt.show()

KeyError: 'In'

In [ ]:
# import numpy as np
# import os  # Import os to create the directory if it doesn't exist
# # # Optional: Verify the data was saved by loading it back
# save_dir = r"D:\Context_Bandit\data"

# loaded_gap_on = np.load(os.path.join(save_dir, 'gap_hb.npy'))

# print("Loaded gap_on:", len(loaded_gap_on))


In [ ]:
# import numpy as np
# import os  # Import os to create the directory if it doesn't exist

# # Define the directory path
# save_dir = r"D:\Context_Bandit\data"


# # Generate some sample data
# gap_on_history = [0.0729385916780978]  + [0.05715130315500683]  + [0.037229248590153974] + [0.02287861606462427]  
# gap_hybrid_history = [0.026531504343849988]  + [0.00580891632373115]  + [0.01207343392775495]
# time_steps = range(1, 61)

# # Convert lists to NumPy arrays
# gap_on_array = np.array(gap_on_history)
# gap_hybrid_array = np.array(gap_hybrid_history)
# time_array = np.array(list(time_steps))

# # Save the arrays to .npy files in the specified directory
# np.save(os.path.join(save_dir, 'gap_on_data.npy'), gap_on_array)
# np.save(os.path.join(save_dir, 'gap_hybrid_data.npy'), gap_hybrid_array)
# np.save(os.path.join(save_dir, 'time_data.npy'), time_array)

# # Optional: Verify the data was saved by loading it back
# loaded_gap_on = np.load(os.path.join(save_dir, 'gap_on_data.npy'))
# loaded_gap_hybrid = np.load(os.path.join(save_dir, 'gap_hybrid_data.npy'))
# loaded_time = np.load(os.path.join(save_dir, 'time_data.npy'))

# print("Loaded gap_on:", loaded_gap_on)
# print("Loaded gap_hybrid:", loaded_gap_hybrid)
# print("Loaded time:", loaded_time)